###GBFS Station Reference Ingestion (US09)

**User Story:** US09 – Data Preprocessing

**Objective:**
Ingest station reference data from the official Bike Share Toronto GBFS feed (station_information) and persist it into Bronze and Silver layers.

**Output:**
Parquet datasets stored in DBFS:
 Bronze: raw station reference dataset
 Silver: clean station dimension table ready for integration

**Task – Ingest and Persist Station Reference Data (GBFS)**
This section extracts station metadata from the GBFS station_information endpoint, validates data quality (nulls, duplicates, schema), and writes the dataset to Bronze and Silver layers for downstream integration.

In [0]:
# =========================
## STEP 1 - CONFIG
# =========================

BRONZE_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/station_id"
SILVER_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/station_id"

#BRONZE_DIR = "dbfs:/Volumes/default/bike_share/Projects/Capstone/data/bronze/station_id"
#SILVER_DIR = "dbfs:/Volumes/default/bike_share/Projects/Capstone/data/silver/station_id"

print("BRONZE_DIR =", BRONZE_DIR)
print("SILVER_DIR =", SILVER_DIR)


In [0]:
# =========================
## STEP 2 – Extract Station Reference Data
# =========================

import requests
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# --- Reusable function (from US06 style) ---
def fetch_gbfs_json(url: str) -> dict:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

# GBFS root used in US06
gbfs_root_url = "https://toronto-us.publicbikesystem.net/customer/gbfs/v2/gbfs.json"

# Get GBFS root
gbfs_root = fetch_gbfs_json(gbfs_root_url)

# Locate station_information endpoint dynamically
feeds = gbfs_root["data"]["en"]["feeds"]

station_info_url = None
for feed in feeds:
    if feed.get("name") == "station_information":
        station_info_url = feed.get("url")
        break

if not station_info_url:
    raise Exception("station_information feed not found.")

print("station_information URL:", station_info_url)

# Fetch station_information JSON
station_info_json = fetch_gbfs_json(station_info_url)
stations = station_info_json["data"]["stations"]

print("Number of stations retrieved:", len(stations))




In [0]:
# =========================
#STEP 3 – Create Structured DataFrame
# =========================

station_schema = StructType([
    StructField("station_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
])

# Keep only required fields (avoid type inference issues)
stations_min = [
    {
        "station_id": str(s.get("station_id")) if s.get("station_id") is not None else None,
        "name": s.get("name"),
        "lat": float(s.get("lat")) if s.get("lat") is not None else None,
        "lon": float(s.get("lon")) if s.get("lon") is not None else None,
    }
    for s in stations
]

df_station_info = spark.createDataFrame(stations_min, schema=station_schema)

print("Rows in DataFrame:", df_station_info.count())
df_station_info.printSchema()

display(df_station_info.limit(10))

In [0]:
# =========================
#STEP 4 – Data Quality Checks
# =========================

from pyspark.sql import functions as F

total_rows = df_station_info.count()

null_station_id = df_station_info.filter(F.col("station_id").isNull()).count()
null_lat_lon = df_station_info.filter(
    F.col("lat").isNull() | F.col("lon").isNull()
).count()

duplicate_station_id = (
    df_station_info
    .groupBy("station_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total rows:", total_rows)
print("Null station_id:", null_station_id)
print("Null lat or lon:", null_lat_lon)
print("Duplicate station_id:", duplicate_station_id)

In [0]:
# ============================
# STEP 5 - Write Bronze and Silver (single file, no partition)
# ============================

# Force single file (dataset is small)
df_station_single = df_station_info.coalesce(1)

# Write Bronze (no partition)
df_station_single.write.mode("overwrite").parquet(BRONZE_DIR)
print("Bronze write completed (single file).")

# Read back Bronze (good practice to ensure consistency)
df_bronze = spark.read.parquet(BRONZE_DIR)

# Write Silver (dimension table copy)
df_bronze.coalesce(1).write.mode("overwrite").parquet(SILVER_DIR)
print("Silver write completed (single file).")

# Verify structure
print("Bronze path content:")
display(dbutils.fs.ls(BRONZE_DIR))

print("Silver path content:")
display(dbutils.fs.ls(SILVER_DIR))

In [0]:
spark.read.parquet(SILVER_DIR).count()